# 🔧 Tuning tham số Naive Bayes & Linear SVM (chạy trên Google Colab)

Notebook này thực hiện toàn bộ việc **thử nhiều giá trị tham số** của 2 mô hình và **chọn ra bộ tham số tốt nhất** (theo Macro F1 trên tập Validation), sau đó **kiểm định trên tập Test** và **tự động cập nhật `train.py`**:

| Mô hình | Tham số thử nghiệm |
|---|---|
| Multinomial Naive Bayes | `alpha` = 0.001 → 5.0 (10 giá trị) |
| Linear SVM (TF-IDF) | `C` = 0.01 → 100 × `class_weight` = {none, balanced} (16 tổ hợp) |
| Linear SVM (Word2Vec) | như trên (16 tổ hợp) |

**Cách lấy dữ liệu (chọn 1 trong 2):**
1. **Tự động** (mặc định): tải bộ VIHSD gốc từ HuggingFace và dựng lại `dataset/processed_data` bằng đúng pipeline của dự án (merge → split stratified → clean_text + tách từ underthesea). ⚠️ Vì máy Colab không có phần `df_*_clean.csv` (không nằm trong git), số liệu tuyệt đối có thể lệch nhẹ so với máy của bạn, nhưng **xu hướng chọn tham số là đáng tin cậy**.
2. **Dùng đúng dataset thật**: nén thư mục `dataset/processed_data` trên máy bạn thành `processed_data.zip` (gồm train.csv, val.csv, test.csv), upload vào tab **Files** bên trái Colab **trước khi chạy** Cell 3 → notebook sẽ tự dùng nó và bỏ qua bước dựng dữ liệu.

**Thời gian chạy:** khoảng 10–20 phút (CPU thường).

➡️ Chạy lần lượt các cell từ trên xuống dưới.

In [ ]:
# ============================================================
# CELL 1: Clone repo dự án từ GitHub (chạy lại cell sẽ tự git pull)
# ============================================================
import os

REPO_URL = "https://github.com/tiudan147/Toxic-classification-for-PTIT--student"
REPO_DIR = "Toxic-classification-for-PTIT--student"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!ls

# Kiểm tra file tuning đã có trên GitHub chưa (nếu chưa push sẽ bị thiếu)
if not os.path.exists("tune_hyperparameters.py"):
    print("\n⚠️ CHƯA CÓ tune_hyperparameters.py trên GitHub!")
    print("   Cách xử lý (chọn 1 trong 2):")
    print("   1. Ở máy local, đẩy file lên repo:  git add tune_hyperparameters.py && git commit -m 'Add tuning script' && git push")
    print("   2. HOẶC upload thẳng file tune_hyperparameters.py vào tab Files bên trái Colab")
    raise SystemExit("Thiếu tune_hyperparameters.py - hãy xử lý rồi chạy lại cell này.")
print("\n[OK] Đã sẵn sàng: repo + tune_hyperparameters.py")

Cloning into 'Toxic-classification-for-PTIT--student'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.


In [ ]:
# CELL 2.1: scikit-learn (chưa thuật toán Naive Bayes, SVM, TF-IDF) + joblib (lưu mô hình)
!pip install -q scikit-learn joblib
import sklearn, joblib
print("[OK] scikit-learn", sklearn.__version__, "| joblib", joblib.__version__)

In [ ]:
# CELL 2.2: pandas + numpy (đọc CSV, xử lý dữ liệu dạng bảng và ma trận)
!pip install -q pandas numpy
import pandas, numpy
print("[OK] pandas", pandas.__version__, "| numpy", numpy.__version__)

In [ ]:
# CELL 2.3: gensim (huấn luyện Word2Vec cho mô hình SVM - Word2Vec)
!pip install -q gensim
import gensim
print("[OK] gensim", gensim.__version__)

In [ ]:
# CELL 2.4: matplotlib + seaborn (vẽ biểu đồ, ma trận nhầm lẫn)
!pip install -q matplotlib seaborn
import matplotlib, seaborn
print("[OK] matplotlib", matplotlib.__version__, "| seaborn", seaborn.__version__)

In [ ]:
# CELL 2.5: underthesea (tách từ tiếng Việt - dùng trong clean_text.py)
!pip install -q underthesea
import underthesea
print("[OK] underthesea", underthesea.__version__)

In [ ]:
# ============================================================
# CELL 3: Lấy dữ liệu
#  - Nếu bạn đã upload processed_data.zip -> tự giải nén và dùng luôn
#  - Ngược lại: tải VIHSD từ HuggingFace và đổi tên cột cho khớp pipeline
# ============================================================
import os, zipfile, urllib.request

# 3a. Ưu tiên dataset thật do người dùng upload lên
if os.path.exists("processed_data.zip") and not os.path.exists("dataset/processed_data/train.csv"):
    print("Tìm thấy processed_data.zip - giải nén...")
    with zipfile.ZipFile("processed_data.zip") as z:
        z.extractall("dataset")
    # Hỗ trợ cả trường hợp zip bọc thêm 1 tầng thư mục
    if not os.path.exists("dataset/processed_data/train.csv"):
        import glob
        found = glob.glob("dataset/**/train.csv", recursive=True)
        assert found, "Không tìm thấy train.csv trong zip!"
        os.makedirs("dataset/processed_data", exist_ok=True)
        for f in glob.glob(os.path.dirname(found[0]) + "/*.csv"):
            os.replace(f, "dataset/processed_data/" + os.path.basename(f))

# 3b. Tự động tải VIHSD từ HuggingFace (bản public, đúng bộ dữ liệu gốc)
if not os.path.exists("dataset/processed_data/train.csv"):
    print("Không có dataset upload - tự động tải VIHSD từ HuggingFace...")
    RAW_DIR = "dataset/raw_vihsd"
    os.makedirs(RAW_DIR, exist_ok=True)
    BASE = "https://huggingface.co/datasets/htdung167/ViHSD/resolve/main"
    files_to_download = {
        "vihsd_train.csv": f"{BASE}/train.csv",
        "vihsd_val.csv":   f"{BASE}/validation.csv",   # dev.csv gốc
        "vihsd_test.csv":  f"{BASE}/test.csv",
    }
    import pandas as pd
    for fname, url in files_to_download.items():
        path = os.path.join(RAW_DIR, fname)
        if not os.path.exists(path):
            print(f"- Tải {fname}...")
            urllib.request.urlretrieve(url, path)
        # Đổi tên cột: free_text -> cmt_col, label_id -> labels (khớp merge_dataset.py)
        df = pd.read_csv(path)
        df = df.rename(columns={"free_text": "cmt_col", "label_id": "labels"})
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"  {fname}: {len(df)} dòng, cột: {list(df.columns)}")
else:
    print("Đã có sẵn dataset/processed_data - dùng đúng dataset này để tuning.")

In [ ]:
# ============================================================
# CELL 4: Dựng dataset/processed_data theo đúng pipeline dự án
#  (merge -> phân chia stratified 70/15/15 -> clean_text + tách từ)
#  Nếu bạn đã upload processed_data thì cell này tự bỏ qua.
# ============================================================
import os

if os.path.exists("dataset/processed_data/train.csv"):
    print("Bỏ qua: dataset/processed_data đã tồn tại.")
else:
    from src.preprocessing_data.merge_dataset import merge_toxic_datasets
    from src.preprocessing_data.split_stratified_data import split_stratified_dataset
    from src.preprocessing_data.process_data import process_splited_data

    print("[1/3] Gộp dữ liệu VIHSD...")
    merge_toxic_datasets(
        dataset_dir="dataset/raw_vihsd",
        output_file="dataset/merged_toxic_dataset.csv"
    )
    print("\n[2/3] Phân chia stratified 70/15/15...")
    split_stratified_dataset(
        input_file="dataset/merged_toxic_dataset.csv",
        output_dir="dataset/splited_raw_data"
    )
    print("\n[3/3] Làm sạch text (teencode, tách từ underthesea)...")
    process_splited_data(
        input_dir="dataset/splited_raw_data",
        output_dir="dataset/processed_data"
    )

In [ ]:
# ============================================================
# CELL 5: CHẠY TUNING - thử toàn bộ lưới tham số và xếp hạng
# ============================================================
!python tune_hyperparameters.py --tune-w2v

In [ ]:
# ============================================================
# CELL 6: Xem tham số tốt nhất + TỰ ĐỘNG cập nhật train.py
# ============================================================
import json, re

best = json.load(open("results/best_params.json", encoding="utf-8"))
print("Bộ tham số tốt nhất (theo Val Macro F1):")
print(json.dumps(best, ensure_ascii=False, indent=2))

code = open("train.py", encoding="utf-8").read()
nb, sv, sw = best["naive_bayes"], best["svm_tfidf"], best.get("svm_w2v")

code, n1 = re.subn(
    r"nb_model = get_naive_bayes_model\([^)]*\)",
    f"nb_model = get_naive_bayes_model(alpha={nb['alpha']})", code)
code, n2 = re.subn(
    r"svm_tfidf_model = get_svm_model\([^)]*\)",
    f"svm_tfidf_model = get_svm_model(C={sv['C']}, class_weight={sv['class_weight']!r})", code)
n3 = 0
if sw:
    code, n3 = re.subn(
        r"svm_w2v_model = get_svm_model\([^)]*\)",
        f"svm_w2v_model = get_svm_model(C={sw['C']}, class_weight={sw['class_weight']!r})", code)

open("train.py", "w", encoding="utf-8").write(code)
print(f"\n[OK] Đã cập nhật train.py (NB: {n1} chỗ, SVM TF-IDF: {n2} chỗ, SVM W2V: {n3} chỗ)")
print("\nCác dòng tham số sau khi sửa:")
for line in code.splitlines():
    if "get_naive_bayes_model(alpha" in line or "get_svm_model(C=" in line:
        print("  ", line.strip())

In [ ]:
# ============================================================
# CELL 7: Tải kết quả về máy (bảng tuning, best_params.json, train.py đã sửa)
# ============================================================
import shutil
from google.colab import files

shutil.make_archive("tuning_results", "zip", "results")
files.download("tuning_results.zip")   # toàn bộ results/ (bảng tham số + báo cáo)
files.download("train.py")            # train.py đã dùng tham số tốt nhất
print("Đã tải xong! Giải nén tuning_results.zip để xem các bảng so sánh tham số.")

## 📌 Sau khi chạy xong

1. Xem bảng xếp hạng tham số ở output **Cell 5** — cột `macro_f1` là chỉ số chọn chính; hàng có `is_baseline=True` là tham số đang dùng để bạn so sánh "có thay đổi nhiều không".
2. Kết quả đầy đủ nằm trong `tuning_results.zip`: `tuning_nb_results.csv`, `tuning_svm_results.csv`, `tuning_svm_w2v_results.csv`, `best_params.json`, `tuning_summary.txt`.
3. `train.py` đã được tự động sửa theo tham số tốt nhất — bạn chỉ cần chép lại vào máy (hoặc commit & push từ Colab):

```python
# Nếu muốn push thẳng từ Colab (thay YOUR_TOKEN bằng GitHub Personal Access Token):
!git config user.email "email-cua-ban@example.com"
!git config user.name "ten-cua-ban"
!git add tune_hyperparameters.py colab_tune_hyperparameters.ipynb train.py results/
!git commit -m "Tuning tham số NB & SVM, cập nhật tham số tốt nhất vào train.py"
!git push https://YOUR_TOKEN@github.com/tiudan147/Toxic-classification-for-PTIT--student.git main
```